<a href="https://colab.research.google.com/github/eunmindow-star/chaeeunmin/blob/main/11_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf

# 1. 구글 드라이브 연결 및 2025년 청소년건강행태조사 원시자료 로드
drive.mount('/content/drive')
df = pd.read_sas('/content/drive/MyDrive/hdac_data/kyrbs2025.sas7bdat')

# 2. 변수 생성 (10주차와 동일한 기준)
df['domain_hs'] = df['GRADE'].isin([4, 5, 6]).astype(int)
df['bf_skip'] = np.where(df['F_BR'].isin([1, 2, 3]), 1,
                         np.where(df['F_BR'].notna(), 0, np.nan))
df['bmi'] = df['WT'] / ((df['HT'] / 100) ** 2)
df['obesity'] = np.where(df['bmi'] >= 25, 1,
                         np.where(df['bmi'].notna(), 0, np.nan))

# 3. 공변량 및 매개변수 결측(무응답 9 등) 정리
# 성별(SEX: 1=남, 2=여 -> 0/1 이분형), 학년(GRADE: 4,5,6), 주관적 경제상태(E_SES: 1~5)
df['sex_bin'] = np.where(df['SEX'] == 1, 1, np.where(df['SEX'] == 2, 0, np.nan))
for col in ['E_SES', 'M_SLP_EN', 'PA_TOT', 'F_NE', 'WC_MN']:
    if col in df.columns:
        df[col] = df[col].replace(9, np.nan)

# 최종 분석 도메인 표시 (고등학생 & 노출·결과·최소충분보정집합 유효응답자)
covs = ['sex_bin', 'GRADE', 'E_SES']
df['in_reg'] = ((df['domain_hs'] == 1) & df[['bf_skip', 'obesity'] + covs].notna().all(axis=1)).astype(int)

print(f"전체 표본: {len(df):,}명 / 회귀분석 도메인 유효 인원: {df['in_reg'].sum():,}명")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
전체 표본: 54,170명 / 회귀분석 도메인 유효 인원: 25,070명


/tmp/ipykernel_27201/2941337115.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['domain_hs'] = df['GRADE'].isin([4, 5, 6]).astype(int)
/tmp/ipykernel_27201/2941337115.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['bf_skip'] = np.where(df['F_BR'].isin([1, 2, 3]), 1,
/tmp/ipykernel_27201/2941337115.py:15: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) ins

In [2]:
# 전역 네임스페이스에서 범주형 지정 함수인 C()와의 충돌 방지를 위해 기존 'C' 변수 삭제
import sys
if 'C' in globals():
    del C

# 도메인 분석용 가중치 설정 (도메인 밖은 0 처리, 도메인 안은 정규화)
df['w_dom'] = np.where(df['in_reg'] == 1, df['W'], 0.0)
df['w_norm'] = df['w_dom'] / df.loc[df['in_reg'] == 1, 'w_dom'].mean()

# GLM 실행을 위해 도메인 유효 행만 추출하되, CLUSTER 표준오차(cov_type='cluster')로 설계 분산 반영
reg_df = df[df['in_reg'] == 1].copy()

# (1) 무보정 모형 (Unadjusted)
m_unadj = smf.glm(
    'obesity ~ bf_skip',
    data=reg_df,
    family=sm.families.Binomial(),
    freq_weights=reg_df['w_norm']
).fit(cov_type='cluster', cov_kwds={'groups': reg_df['CLUSTER']})

# (2) 최소충분보정집합 보정 모형 (Adjusted: SEX, GRADE, E_SES)
m_adj = smf.glm(
    'obesity ~ bf_skip + C(sex_bin) + C(GRADE) + C(E_SES)',
    data=reg_df,
    family=sm.families.Binomial(),
    freq_weights=reg_df['w_norm']
).fit(cov_type='cluster', cov_kwds={'groups': reg_df['CLUSTER']})

# (3) 매개변수 포함 과보정 모형 (Overcontrolled: M_SLP_EN, PA_TOT, F_NE 추가)
over_vars = [v for v in ['M_SLP_EN', 'PA_TOT', 'F_NE'] if v in reg_df.columns]
reg_over_df = reg_df.dropna(subset=over_vars).copy()
reg_over_df['w_norm_over'] = reg_over_df['W'] / reg_over_df['W'].mean()

formula_over = 'obesity ~ bf_skip + C(sex_bin) + C(GRADE) + C(E_SES) + ' + ' + '.join([f'C({v})' for v in over_vars])
m_over = smf.glm(
    formula_over,
    data=reg_over_df,
    family=sm.families.Binomial(),
    freq_weights=reg_over_df['w_norm_over']
).fit(cov_type='cluster', cov_kwds={'groups': reg_over_df['CLUSTER']})

# 결과 요약표 출력 함수
def summarize_or(model, model_name):
    coef = model.params['bf_skip']
    se = model.bse['bf_skip']
    pval = model.pvalues['bf_skip']
    or_val = np.exp(coef)
    ci_low = np.exp(coef - 1.96 * se)
    ci_high = np.exp(coef + 1.96 * se)
    return {
        '모형': model_name,
        'OR': f"{or_val:.2f}",
        '95% CI': f"{ci_low:.2f} - {ci_high:.2f}",
        'p-value': f"{pval:.4f}"
    }

res_table = pd.DataFrame([
    summarize_or(m_unadj, '1. Unadjusted (무보정)'),
    summarize_or(m_adj, '2. Adjusted (최소충분보정집합)'),
    summarize_or(m_over, '3. Overcontrolled (매개변수 포함)')
])

print("=== [Table 2] 아침식사 결식과 비만 유병의 복합표본 로지스틱 회귀분석 ===")
display(res_table)

/tmp/ipykernel_27201/4294406440.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['w_dom'] = np.where(df['in_reg'] == 1, df['W'], 0.0)
/tmp/ipykernel_27201/4294406440.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['w_norm'] = df['w_dom'] / df.loc[df['in_reg'] == 1, 'w_dom'].mean()
/usr/local/lib/python3.13/dist-packages/statsmodels/genmod/generalized_linear_model.py:1488: SpecificationWarning: cov_type not fully supported with freq_weights
  glm_results = GLMResults(
/usr/local/lib/python3.13/dist-packages/statsmode

=== [Table 2] 아침식사 결식과 비만 유병의 복합표본 로지스틱 회귀분석 ===


/usr/local/lib/python3.13/dist-packages/statsmodels/genmod/generalized_linear_model.py:1488: SpecificationWarning: cov_type not fully supported with freq_weights
  glm_results = GLMResults(


,모형,OR,95% CI,p-value
0,1. Unadjusted (무보정),1.07,1.00 - 1.15,0.0564
1,2. Adjusted (최소충분보정집합),1.09,1.01 - 1.17,0.0211
2,3. Overcontrolled (매개변수 포함),1.10,1.02 - 1.19,0.0124
